# 作业二：定制化词典流水线（结巴 + 词向量 + K 近邻 + TF/DF）

在仓库根目录将 `homework_2` 加入路径后运行下列单元；或直接运行 `python homework_2/main.py` 生成 `output/` 下全部产物。

In [ ]:
import os, sys
from pathlib import Path

cwd = Path.cwd().resolve()
HW2 = cwd if cwd.name == 'homework_2' else cwd / 'homework_2'
if not (HW2 / 'main.py').is_file():
    raise RuntimeError('请在仓库根目录或 homework_2 目录下打开本 Notebook')
sys.path.insert(0, str(HW2))
os.chdir(HW2)
print('工作目录:', os.getcwd())

In [ ]:
from corpus_collector import build_mixed_corpus, save_raw
from config import OUTPUT_DIR

raw = build_mixed_corpus()
save_raw(raw, 'mixed_raw_corpus.txt')
print('语料长度:', len(raw))

In [ ]:
from config import STOPWORDS_PATH, SEED_AI, SEED_FINANCE
from pipeline import run_pipeline, compare_demo_sentences
import os

user_ai = os.path.join(OUTPUT_DIR, 'user_dict_ai.txt')
user_fin = os.path.join(OUTPUT_DIR, 'user_dict_finance.txt')
r_ai = run_pipeline(raw, STOPWORDS_PATH, SEED_AI, user_ai, 'AI')
r_fin = run_pipeline(raw, STOPWORDS_PATH, SEED_FINANCE, user_fin, '财经')
print('\n'.join(r_ai['log'][-4:]))

In [ ]:
from visualize import plot_dict_growth, plot_token_freq_segmented

n = min(len(r_ai['history_sizes']), len(r_fin['history_sizes']))
plot_dict_growth(r_ai['history_sizes'][:n], r_fin['history_sizes'][:n])
plot_token_freq_segmented(r_ai['final_tokenized'], fname='token_freq_ai_dict.png')
plot_token_freq_segmented(r_fin['final_tokenized'], fname='token_freq_finance_dict.png')

In [ ]:
import os
from IPython.display import Image, display

for fn in ['dict_growth.png', 'token_freq_ai_dict.png', 'token_freq_finance_dict.png']:
    p = os.path.join(OUTPUT_DIR, fn)
    if os.path.isfile(p):
        display(Image(filename=p))

## 结果解读（分析要点）

1. **语料结构**：若百度百科抓取为空，主体语料来自百度资讯 + `homework_1/ai_corpus.txt`；财经相关段落相对较少时，**词向量近邻**更偏「通用高频词」，财经词典迭代容易提前停止。
2. **AI 词典 31→202**：多轮 K 近邻 + TF/DF 每轮仍会并入一批词；若需降低噪声，可提高 `config.py` 中 `MIN_TF` / `MIN_DF` 或减少每轮并入数量（`pipeline.py` 中 `accepted[:32]`）。
3. **财经词典 33→34**：第二轮即无满足 TF/DF 的新词，符合「语料与种子不匹配则扩展慢」的预期；可增加财经百科/研报类语料或略放宽 `MIN_DF` 做对照实验。
4. **分词对照**：例句中「预训练」被切成「训练」、以及「逆回购」被切成「回购」，说明**用户词典未包含或未压倒默认路径**；可将「预训练」「逆回购」作为整词写入种子或提高其用户词典词频并重新跑迭代。
5. **环境差异**：未安装 Gensim 时使用 **NumPy 共现+PPMI+SVD** 近似词向量，近邻语义弱于 Word2Vec；若本机可装 **Gensim + Annoy**，结果更接近课件示例。